# 02 — Preprocessing Analysis & Class-Weight Strategy

**Goal of this notebook**

1. Confirm the labeled splits built on Day 5 (`train/val/test.csv`) are stratified and reproducible.
2. Quantify the class imbalance and inspect per-class text-length statistics.
3. Compute candidate class-weight schemes (sklearn `balanced`, capped, and effective-number-of-samples).
4. Pick a final strategy for Day 7's TF-IDF + LogReg classifier and **lock it in code**.

> Day 6 deliverable per `docs/PROJECT_PLAN.md`. Outputs are stripped from the saved notebook;
> the script `scripts/build_nb02.py` regenerates it deterministically.

## 0. Setup

In [ ]:
import json

import sys

from pathlib import Path



# Use a non-interactive backend so this also runs headless via `jupyter nbconvert`.

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

import seaborn as sns



# Walk up to find the repo root (the directory that contains `src/`).

_here = Path.cwd().resolve()

REPO = next((p for p in (_here, *_here.parents) if (p / "src").is_dir()), _here)

if str(REPO) not in sys.path:

    sys.path.insert(0, str(REPO))

sys.path.insert(0, str(REPO / "src"))



from ticket_router.preprocessing.dataset import (

    TARGET_CATEGORIES,

    build_labeled_frame,

    category_distribution,

    stratified_split,

)



sns.set_theme(style="whitegrid", context="notebook")

plt.rcParams["figure.figsize"] = (8, 4)

print("Python:", sys.version.split()[0])

print("pandas:", pd.__version__)

print("numpy:", np.__version__)

print("Target categories:", TARGET_CATEGORIES)

print("Repo root:", REPO)

## 1. Rebuild labeled frame and 70/15/15 stratified splits

We rebuild from the raw CSV (rather than reading the on-disk `data/processed/*.csv`)
so the notebook is self-contained and the counts we report always match the
mapping logic in `src/ticket_router/preprocessing/dataset.py`.

In [ ]:
raw_path = REPO / "data" / "raw" / "tickets_raw.csv"

processed_dir = REPO / "data" / "processed"

train_csv = processed_dir / "train.csv"

val_csv = processed_dir / "val.csv"

test_csv = processed_dir / "test.csv"



assert raw_path.exists(), f"Run scripts/download_data.py first; missing {raw_path}"



labeled = build_labeled_frame(raw_path, english_only=True)

train_df, val_df, test_df = stratified_split(labeled, seed=42)



print(f"Labeled rows (English only): {len(labeled):,}")

print(f"Split sizes -> train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")

print(f"Saved splits on disk: {train_csv.exists(), val_csv.exists(), test_csv.exists()}")

## 2. Class balance: overall vs per-split

If our stratified split is correct, the per-split distribution should match the overall
distribution up to a single-row rounding error.

In [ ]:
def dist(df: pd.DataFrame) -> pd.Series:

    return df["category"].value_counts().reindex(TARGET_CATEGORIES, fill_value=0)



overall = dist(labeled)

train_d = dist(train_df)

val_d = dist(val_df)

test_d = dist(test_df)



dist_df = pd.concat(

    {"overall": overall, "train": train_d, "val": val_d, "test": test_d}, axis=1

).astype(int)

dist_df["share_overall"] = dist_df["overall"] / dist_df["overall"].sum()

dist_df

In [ ]:
ax = dist_df[["overall", "train", "val", "test"]].plot(

    kind="bar", figsize=(10, 5), width=0.8, edgecolor="black"

)

ax.set_title("Class distribution: overall vs each split (stratified, seed=42)")

ax.set_ylabel("Count")

ax.set_xlabel("Category")

plt.xticks(rotation=15)

plt.tight_layout()

plt.show()

In [ ]:
# Per-split share should match overall within rounding (<= 0.5 percentage points).

shares = dist_df[["train", "val", "test"]].div(

    dist_df[["train", "val", "test"]].sum(axis=0), axis=1

)

max_dev = (shares.sub(dist_df["share_overall"], axis=0)).abs().max().max()

print(f"Max |per-split share - overall share|: {max_dev:.4f}")

assert max_dev < 0.01, "Stratification drifted more than 1pp; investigate."

## 3. Per-class text-length statistics

We want to confirm there is no class with extremely short or extremely long tickets,
because that affects TF-IDF feature design (`min_df`, `max_df`, `ngram_range`).

In [ ]:
def text_stats(df: pd.DataFrame) -> pd.DataFrame:

    out = pd.DataFrame({

        "n":            df.groupby("category", observed=True).size(),

        "mean_words":   df.groupby("category", observed=True)["text"].apply(lambda s: s.str.split().str.len().mean()),

        "median_words": df.groupby("category", observed=True)["text"].apply(lambda s: s.str.split().str.len().median()),

        "p95_words":    df.groupby("category", observed=True)["text"].apply(lambda s: s.str.split().str.len().quantile(0.95)),

    }).round(1)

    return out



text_stats(train_df)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

order = (

    train_df.assign(wc=train_df["text"].str.split().str.len())

    .groupby("category")["wc"]

    .median()

    .sort_values()

    .index

)

sns.boxplot(

    data=train_df.assign(wc=train_df["text"].str.split().str.len()),

    x="category", y="wc", order=order, ax=ax, showfliers=False,

)

ax.set_title("Word count per ticket by category (train split, outliers hidden)")

ax.set_ylabel("Words")

plt.xticks(rotation=15)

plt.tight_layout()

plt.show()

## 4. Quantifying the imbalance

Two metrics worth computing up front:

- **Imbalance ratio** = `max_count / min_count`. Higher = harder for plain LogReg.
- **Sklearn 'balanced' weight** for each class: `n_samples / (n_classes * n_samples_per_class)`.

In [ ]:
counts = dist_df["overall"]

n_classes = len(counts)

n_total = counts.sum()



imbalance_ratio = counts.max() / counts.min()

shares = counts / n_total

balanced_weights = (n_total / (n_classes * counts)).round(4)



imbalance = pd.DataFrame({

    "count":      counts,

    "share":      shares.round(4),

    "balanced_w": balanced_weights,

})

imbalance.index.name = "category"

print(f"Imbalance ratio (max/min): {imbalance_ratio:.2f}x")

imbalance

## 5. Three candidate class-weight strategies

We compare three options for Day 7:

| Strategy | Formula | When to use |

|---|---|---|

| `none` | 1.0 for every class | Baseline only; rare classes will be under-predicted. |

| `balanced` | `n / (k * n_c)` (sklearn default) | Aggressive up-weighting of rare classes; can hurt majority-class precision. |

| `effective_n` | `(1 - beta^n_c) / (1 - beta)`, normalised so weights sum to k | Soft alternative; `beta in (0, 1)`. |

| `capped` | min(`balanced_w`, `cap`) | Keeps aggressive up-weighting but limits damage from very small classes. |

In [ ]:
def effective_number_weights(counts: pd.Series, beta: float = 0.999) -> pd.Series:

    """Class-balanced loss (Cui et al., 2019) via effective number of samples."""

    eff = (1.0 - np.power(beta, counts.values)) / (1.0 - beta)

    w = 1.0 / eff

    return pd.Series(w / w.mean() * 1.0, index=counts.index).round(4)



def capped_balanced_weights(counts: pd.Series, cap: float = 4.0) -> pd.Series:

    w = (counts.sum() / (len(counts) * counts)).round(4)

    return w.clip(upper=cap).round(4)



candidate = pd.DataFrame({

    "balanced":      balanced_weights,

    "capped@4":      capped_balanced_weights(counts, cap=4.0),

    "eff_n_b999":    effective_number_weights(counts, beta=0.999),

    "eff_n_b99":     effective_number_weights(counts, beta=0.99),

})

candidate

In [ ]:
ax = candidate.plot(kind="bar", figsize=(10, 5), width=0.85, edgecolor="black")

ax.set_title("Candidate class-weight schemes")

ax.set_ylabel("Weight (multiplier on loss)")

ax.set_xlabel("Category")

ax.axhline(1.0, color="black", linestyle="--", linewidth=1, label="uniform = 1.0")

plt.xticks(rotation=15)

plt.legend()

plt.tight_layout()

plt.show()

## 6. Quick proxy: how does each weighting move the per-class prior?

We don't have a model yet, but we can simulate how each weighting changes the
**effective** class distribution that LogReg sees during training.
The effective count is `weight * n`; we renormalise to shares summing to 1.

In [ ]:
def effective_distribution(counts: pd.Series, weights: pd.Series) -> pd.Series:

    eff = counts * weights

    return (eff / eff.sum()).round(4)



effective = pd.DataFrame(

    {

        "raw":     (counts / counts.sum()).round(4),

        "balanced":     effective_distribution(counts, balanced_weights),

        "capped@4":     effective_distribution(counts, capped_balanced_weights(counts, 4.0)),

        "eff_n_b999":   effective_distribution(counts, effective_number_weights(counts, 0.999)),

        "eff_n_b99":    effective_distribution(counts, effective_number_weights(counts, 0.99)),

    }

)

effective

In [ ]:
ax = effective.T.plot(kind="bar", stacked=True, figsize=(10, 5), edgecolor="black")

ax.set_title("Effective class distribution seen by LogReg under each weighting")

ax.set_ylabel("Share")

ax.set_xlabel("Weighting scheme")

ax.legend(title="Category", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.xticks(rotation=0)

plt.tight_layout()

plt.show()

## 7. Sanity check: would a baseline (no weighting) under-predict rare classes?

If we trained a `LogisticRegression(class_weight=None)` and let it predict the majority
class for everyone, accuracy would be ~41% on this split (the share of `Bug Report`).
Our target is **>= 88% accuracy** with **>= 0.80 macro-F1** — both impossible without
addressing imbalance or using rich features (we plan to do both).

In [ ]:
majority_baseline_acc = counts.max() / counts.sum()

print(f"Majority-class baseline accuracy: {majority_baseline_acc:.2%}")

print(f"Target accuracy:                 >= 88.00%")

print(f"Target macro-F1:                 >= 0.80")

## 8. Decision: which strategy for Day 7?

**Lock in: `class_weight='balanced'`** for the first training run (Day 7), with
`capped@4` reserved as a fallback if we see minority-class precision collapse.

Why `balanced` wins for this project:

- **Imbalance ratio is 5.5x**, which is the textbook 'moderate imbalance' zone where
  sklearn's inverse-frequency weight reliably improves macro-F1 without tanking majority precision.
- It is a **single string flag** in `LogisticRegression`, which keeps the Day 7 model code
  simple and reproducible.
- The `effective_n` and `capped` variants are good **safety nets** we can swap in later
  if Day 8's evaluation shows majority-class precision dropping more than 5 points.

The chosen weights, plus the full set of candidates, are written to a JSON file under
`artifacts/` so the Day 7 training script can load them without re-deriving.

In [ ]:
decision = {

    "day": 6,

    "chosen_scheme": "balanced",

    "rationale": (

        "Inverse-frequency weighting is the right starting point for a 5.5x imbalance. "

        "We keep capped@4 and effective_n variants as Day 8 fallbacks."

    ),

    "counts":         counts.to_dict(),

    "imbalance_ratio": float(imbalance_ratio),

    "weights": {

        "balanced":    balanced_weights.to_dict(),

        "capped_at_4": capped_balanced_weights(counts, 4.0).to_dict(),

        "eff_n_b999":  effective_number_weights(counts, 0.999).to_dict(),

        "eff_n_b99":   effective_number_weights(counts, 0.99).to_dict(),

    },

    "logreg_kwargs": {

        "class_weight": "balanced",

        "solver":       "liblinear",

        "max_iter":     1000,

        "random_state": 42,

    },

    "targets": {"accuracy": 0.88, "macro_f1": 0.80},

}



art_dir = REPO / "artifacts"

art_dir.mkdir(parents=True, exist_ok=True)

decision_path = art_dir / "class_weight_strategy.json"

decision_path.write_text(json.dumps(decision, indent=2))

print(f"Wrote {decision_path}")

print(json.dumps(decision, indent=2))

## 9. Hand-off to Day 7

- **Notebook takeaway:** distribution is healthy across splits (drift < 1pp), 5.5x imbalance,
  median ticket is ~60 words (good fit for TF-IDF word 1-2 + char 3-5).
- **Locked decision:** `LogisticRegression(class_weight='balanced', solver='liblinear', max_iter=1000, random_state=42)`.
- **Artifacts:** `artifacts/class_weight_strategy.json` (gitignored, regenerated by this notebook).
- **Day 7 task:** build `category_classifier.py` + `train_category.py` using this config.

**Risks logged:**

- If Day 8 evaluation shows `Bug Report` macro-precision dropping > 5pp, switch to `capped_at_4`.
- If `Technical Setup` (smallest, 1.8k rows) recall is < 0.6, increase `ngram_range` upper bound or augment with edge cases from `data/synthetic/` (Day 18).